The DataFrame is Spark's primary high-level API for structured data. It combines the familiarity of a table — named columns, types, SQL-style operations — with the distributed execution model of Spark. Understanding DataFrames deeply is the single biggest skill for the Databricks Associate exam and real-world Spark work.

## The Spark API Hierarchy

![Spark API Hierarchy](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-api-hierarchy.svg)

- **SQL** and **DataFrame/Dataset** both go through Catalyst and Tungsten — they produce identical execution plans
- **RDD** bypasses Catalyst — no automatic optimisation
- In Python, `DataFrame` is the only option; `Dataset[T]` (typed) is Scala/Java only
- `DataFrame` is an alias for `Dataset[Row]`

## DataFrame Structure

A DataFrame is a **distributed table** — it has a schema (column names and types) and rows, partitioned across executors.

**Schema** (applies to every partition):

| id `LongType` | name `StringType` | age `IntegerType` | active `BooleanType` |
|---|---|---|---|

**Partition 1** — Executor 1 (rows 1–4):

| id | name  | age | active |
|----|-------|-----|--------|
| 1  | Alice | 30  | true   |
| 2  | Bob   | 25  | false  |
| 3  | Carol | 35  | true   |
| 4  | Dan   | 28  | true   |

**Partition 2** — Executor 2 (rows 5–8):

| id | name  | age | active |
|----|-------|-----|--------|
| 5  | Eve   | 22  | false  |
| 6  | Frank | 40  | true   |
| 7  | Grace | 31  | true   |
| 8  | Hank  | 29  | false  |

> Each partition holds a **subset of rows** with **all columns** — partitioning is always horizontal (by row), never by column.


Key properties:
- **Immutable** — transformations return a new DataFrame; the original is unchanged
- **Lazily evaluated** — transformations build a plan; actions execute it
- **Schema-enforced** — every row matches the same column types
- **Partitioned** — rows are split across partitions for parallel processing

## The Schema — StructType & StructField

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, BooleanType,
    DoubleType, DateType, TimestampType, ArrayType, MapType
)

spark = (
    SparkSession.builder
    .appName("DataFrames")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

# Explicit schema for a customers table
schema = StructType([
    StructField("customer_id",  StringType(),   nullable=False),
    StructField("full_name",    StringType(),   nullable=False),
    StructField("city",         StringType(),   nullable=True),
    StructField("credit_score", IntegerType(),  nullable=True),
    StructField("kyc_verified", BooleanType(),  nullable=False),
])

print(schema)


In [ ]:
# Spark also accepts DDL string syntax — often more concise
schema_ddl = "customer_id STRING NOT NULL, full_name STRING NOT NULL, city STRING, credit_score INT, kyc_verified BOOLEAN NOT NULL"

print(spark.createDataFrame([], schema_ddl).schema)


## Creating DataFrames

In [ ]:
# 1. From a Python list with explicit schema
data = [
    ("CUST0001", "Aarav Sharma",  "Mumbai",    780, True),
    ("CUST0002", "Priya Nair",    "Delhi",     650, True),
    ("CUST0003", "Rohan Gupta",   "Bengaluru", 720, False),
    ("CUST0004", "Anjali Mehta",  "Pune",      810, True),
    ("CUST0005", "Vikram Reddy",  "Hyderabad", 590, True),
]
df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()


In [ ]:
# 2. Schema inference (convenient, but slower — Spark scans the data)
df_inferred = spark.createDataFrame(data, ["customer_id", "full_name", "city", "credit_score", "kyc_verified"])
df_inferred.printSchema()  # credit_score may be inferred as bigint instead of int


In [ ]:
# 3. spark.range — fast way to create integer sequences for testing
spark.range(5).show()
spark.range(0, 20, step=4).show()  # start, end, step

In [ ]:
# 4. From a pandas DataFrame
import pandas as pd

pdf = pd.DataFrame({"x": [10, 20, 30], "y": ["a", "b", "c"]})
df_from_pandas = spark.createDataFrame(pdf)
df_from_pandas.show()

## Exploring a DataFrame

In [ ]:
print("Shape       :", df.count(), "rows x", len(df.columns), "cols")
print("Columns     :", df.columns)
print("dtypes      :", df.dtypes)
print("Partitions  :", df.rdd.getNumPartitions())
df.describe("credit_score").show()   # count, mean, stddev, min, max


## Selecting Columns

There are several ways to reference a column — they are interchangeable but have subtle differences:

In [ ]:
from pyspark.sql.functions import col, expr

# Four equivalent ways to reference a column
df.select("full_name")           # string name
df.select(col("full_name"))      # Column object — preferred when applying functions
df.select(df["full_name"])       # DataFrame attribute — ties the column to this df
df.select(df.full_name)          # dot syntax — convenient but breaks with spaces/reserved words

# Select multiple columns
df.select("customer_id", "full_name", "city").show()

# expr() — use SQL expression strings inline
df.select(expr("credit_score + 20 AS adjusted_score")).show()


## Adding, Renaming, and Dropping Columns

In [ ]:
from pyspark.sql.functions import col, lit, upper, when

# withColumn — add or replace a column
df2 = (
    df
    .withColumn("tier",       when(col("credit_score") >= 750, "prime").otherwise("standard"))
    .withColumn("name_upper", upper(col("full_name")))
    .withColumn("bank",       lit("Fintech Bank"))     # literal constant
)
df2.show()

# withColumnRenamed
df2.withColumnRenamed("name_upper", "FULL_NAME").show(truncate=False)

# drop
df2.drop("bank", "name_upper").show()


## Filtering Rows

In [ ]:
# filter() and where() are identical
df.filter(col("credit_score") > 700).show()
df.where("credit_score > 700").show()   # SQL string also works

# Multiple conditions — use & (AND), | (OR), ~ (NOT)
# Always wrap each condition in parentheses!
df.filter((col("credit_score") > 650) & (col("kyc_verified") == True)).show()
df.filter((col("credit_score") < 650) | (col("credit_score") > 750)).show()

# isin — filter by city list
df.filter(col("city").isin("Mumbai", "Delhi")).show()

# NOT isin
df.filter(~col("city").isin("Mumbai", "Delhi")).show()


## Handling Nulls

In [ ]:
from pyspark.sql.functions import coalesce, col, lit

# Create a DataFrame with nulls — realistic: some customers have missing data
df_nulls = spark.createDataFrame(
    [
        ("CUST0001", "Aarav Sharma",  780),
        ("CUST0002", None,            None),
        ("CUST0003", "Rohan Gupta",   720),
    ],
    ["customer_id", "full_name", "credit_score"]
)

# Filter nulls
df_nulls.filter(col("full_name").isNull()).show()      # customers with missing name
df_nulls.filter(col("full_name").isNotNull()).show()   # customers with a name

# Drop rows containing any null
df_nulls.dropna().show()
df_nulls.dropna(subset=["full_name"]).show()   # only check specific columns

# Fill nulls
df_nulls.fillna({"full_name": "Unknown", "credit_score": 0}).show()

# coalesce — return first non-null value
df_nulls.select(coalesce(col("full_name"), lit("N/A")).alias("name_safe")).show()


## Type Casting

In [ ]:
from pyspark.sql.types import DoubleType

# cast() — convert credit_score to DoubleType (e.g. for ML feature vectors)
df.withColumn("credit_score_d",   col("credit_score").cast(DoubleType())).printSchema()

# DDL string form — often more readable
df.withColumn("credit_score_str", col("credit_score").cast("string")).printSchema()


## Sorting and Deduplication

In [ ]:
from pyspark.sql.functions import desc

# orderBy / sort (synonyms)
df.orderBy(col("credit_score").desc()).show()
df.sort("credit_score", ascending=False).show()
df.orderBy(col("kyc_verified").desc(), col("credit_score").asc()).show()  # multi-column

# distinct — removes duplicate rows (causes a shuffle)
dupes = spark.createDataFrame(
    [
        ("CUST0001", "Aarav Sharma",  "Mumbai"),
        ("CUST0001", "Aarav Sharma",  "Mumbai"),   # exact duplicate
        ("CUST0002", "Priya Nair",    "Delhi"),
    ],
    ["customer_id", "full_name", "city"]
)
dupes.distinct().show()

# dropDuplicates — deduplicate on specific columns only
dupes.dropDuplicates(["customer_id"]).show()


## Converting Between APIs

In [ ]:
# DataFrame -> pandas (collects all data to driver — only for small results)
pdf = df.toPandas()
print(type(pdf))
print(pdf)

# DataFrame -> RDD
rdd = df.rdd          # RDD of Row objects
print(rdd.first())

# RDD -> DataFrame
from pyspark.sql import Row
rdd2 = spark.sparkContext.parallelize([
    Row(customer_id="CUST0006", full_name="Sunita Patel", city="Chennai", credit_score=700, kyc_verified=True)
])
df_from_rdd = spark.createDataFrame(rdd2)
df_from_rdd.show()


## Summary

- A **DataFrame** is a distributed table with a schema — immutable, lazily evaluated, and partitioned across executors.
- SQL, DataFrame, and Dataset APIs all go through **Catalyst** and produce identical execution plans. RDD bypasses Catalyst.
- Define schemas with `StructType` / `StructField`, or as a DDL string. Explicit schemas are faster and safer than inferred ones.
- Reference columns with strings, `col()`, `df["col"]`, or `expr()` — `col()` is preferred when applying functions.
- `withColumn` adds or replaces; `withColumnRenamed` renames; `drop` removes.
- `filter` / `where` are identical; combine conditions with `&`, `|`, `~` — always parenthesise each condition.
- Handle nulls with `isNull`, `isNotNull`, `dropna`, `fillna`, and `coalesce`.
- `toPandas()` collects to the driver — only use on small DataFrames. `df.rdd` gives the underlying RDD when needed.